<a href="https://colab.research.google.com/github/Pankaj429w63/Affectra-AI/blob/main/training/notebooks/AffectraAI_MELD_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Affectra AI — Multimodal Emotion Intelligence Training

**Dataset:** MELD (Multimodal EmotionLines Dataset)  
**Model:** Gated Multimodal Fusion (text + audio + video → emotion + sentiment)  
**Environment:** Google Colab free GPU (T4)

---

## Before You Begin

1. **Enable GPU**: Runtime → Change runtime type → Hardware accelerator → **GPU** → Save
2. **Run cells one by one** — do NOT click 'Run all'. Each section has important notes.
3. **Training order**: Sections 01–09 are setup + validation. Section 09 ends with a **STOP** before full training.
4. **Google Drive**: Mount it early — feature caches (~3 GB) are saved there so they survive Colab resets.

---

## Notebook Sections

| Section | What It Does |
|---|---|
| 01 | Mount Google Drive |
| 02 | Verify GPU |
| 03 | Clone Affectra AI repository |
| 04 | Install dependencies |
| 05 | Download MELD dataset (Colab only) |
| 06 | Extract and inspect MELD |
| 07 | Download official MELD annotations |
| 08 | Validate dataset |
| 09 | Run smoke test → **STOP for confirmation** |
| 10 | Extract and cache all features (text / audio / video) |
| 11 | Train fusion model |
| 12 | Evaluate development split |
| 13 | Final test evaluation (run ONCE) |
| 14 | Export final model artifacts |


---
## Section 01 — Mount Google Drive

**Why:** Feature caches (~3 GB) and model checkpoints are saved to Google Drive.  
Without this, everything is lost when the Colab session resets.

**What to do:** Run this cell. A popup will ask you to authorise Drive access.  
Click the link, choose your Google account, and paste the authorisation code.

In [ ]:
# Mount Google Drive so we can save caches and checkpoints permanently
from google.colab import drive
drive.mount('/content/drive')

import os

# Create the Affectra AI folder structure on Drive
DRIVE_ROOT = '/content/drive/MyDrive/AffectraAI'
for subdir in ['checkpoints', 'feature_cache', 'logs', 'training_outputs']:
    os.makedirs(f'{DRIVE_ROOT}/{subdir}', exist_ok=True)
    print(f'Created: {DRIVE_ROOT}/{subdir}')

print('\n✅ Google Drive mounted and directories created.')

---
## Section 02 — Verify GPU

**Why:** Training on CPU would take days. We need a GPU (T4, L4, or A100).  
If this cell shows 'No GPU', go to Runtime → Change runtime type → GPU.

In [ ]:
import torch

print('=== GPU / Hardware Status ===')
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'GPU: {gpu}')
    print(f'VRAM: {vram:.1f} GB')
    print('✅ GPU is available — training will be fast!')
else:
    print('❌ No GPU detected!')
    print('   Go to: Runtime → Change runtime type → GPU → Save')
    print('   Then re-run this cell.')
    raise SystemExit('Please enable GPU before continuing.')

print(f'\nPyTorch version: {torch.__version__}')
print(f'CUDA version:    {torch.version.cuda}')

---
## Section 03 — Clone Affectra AI Repository

**Why:** All training code lives in `training/src/`. We need to clone the repository  
so we can import the Python modules.

**Note:** If you already cloned in a previous session, the `git pull` will update it.

In [ ]:
import os
import subprocess

REPO_URL = 'https://github.com/Pankaj429w63/Affectra-AI.git'
REPO_DIR = '/content/Affectra-AI'

if os.path.exists(REPO_DIR):
    print(f'Repository already cloned at {REPO_DIR}. Pulling latest...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    print(f'Cloning repository to {REPO_DIR}...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

# Add the repository root to Python's import path
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'\n✅ Repository ready at: {REPO_DIR}')

# Verify training/src is importable
import importlib
try:
    importlib.import_module('training.src.config')
    print('✅ training.src.config importable')
except ImportError as e:
    print(f'❌ Import failed: {e}')
    print('   Make sure the repo was cloned correctly.')

---
## Section 04 — Install Dependencies

**Why:** Colab has PyTorch pre-installed but we need additional packages:  
Hugging Face Transformers, soundfile, librosa, and opencv.  

**Expected time:** 2–4 minutes

In [ ]:
# Install all required packages from the requirements file
# This runs pip quietly (-q) — increase verbosity by removing -q if you need to debug
!pip install -q -r /content/Affectra-AI/training/requirements-colab.txt

# sentencepiece is sometimes needed by HuggingFace tokenizers
!pip install -q sentencepiece

print('\n✅ Dependencies installed.')

# Verify key imports
import transformers
import librosa
import cv2
import soundfile
import sklearn

print(f'transformers: {transformers.__version__}')
print(f'librosa:      {librosa.__version__}')
print(f'opencv:       {cv2.__version__}')
print(f'scikit-learn: {sklearn.__version__}')

---
## Section 05 — Download MELD Dataset

**Why:** The MELD dataset is ~11 GB. We download it directly into Colab storage —  
**never onto your local Windows machine**.

**Expected time:** 10–25 minutes depending on Colab's network speed.

**Important:** If this cell was already run in a previous session and the file exists,  
it will skip the download automatically.

In [ ]:
# Section 05 — Download the MELD dataset into Colab (~10.1 GB)
#
# This calls the tested downloader in training/src/download.py, which:
#   - tries each mirror in config.MELD_RAW_URLS IN ORDER until one works
#     (so if one host is unreachable from Colab, it auto-falls back),
#   - verifies each URL, downloads to a .part file, checks the size,
#     and confirms it is a valid .tar.gz before keeping it,
#   - skips the download if the archive already exists (safe to re-run).
#
# The mirror list lives ONCE in training/src/config.py (MELD_RAW_URLS),
# so the notebook and the code can never disagree on it.
from training.src.download import download_meld_archive

archive_path = download_meld_archive(force=False)
print(f'
✅ MELD archive ready at: {archive_path}')

---
## Section 06 — Extract MELD and Inspect Structure

**Why:** We extract the archive and then inspect the exact folder structure  
that was created. We do NOT assume a hard-coded path — the code  
searches for the CSV files automatically.

**Expected time:** 3–8 minutes

In [ ]:
import os
from training.src.download import (
    extract_meld_archive,
    inspect_extracted_structure,
    find_meld_root,
    find_video_dir,
)

ARCHIVE_PATH = '/content/MELD.Raw.tar.gz'
DATA_DIR     = '/content/meld_data' # This will contain MELD.Raw/

# ── Extract main archive ──────────────────────────────────────────────────
# Skips if already extracted
print('Extracting MELD main archive (MELD.Raw.tar.gz)...')
# This extracts MELD.Raw.tar.gz into DATA_DIR, resulting in DATA_DIR/MELD.Raw/
extract_meld_archive(ARCHIVE_PATH, force=False)

# Path to the MELD.Raw directory (where the nested archives are)
MELD_RAW_DIR = os.path.join(DATA_DIR, 'MELD.Raw')

# ── Extract nested archives (train, dev, test) ────────────────────────────
print('\nExtracting nested MELD split archives (train.tar.gz, etc.)...')
# This will extract train.tar.gz into MELD_RAW_DIR/train/, etc.
splits_to_extract = ['train', 'dev', 'test']
for split in splits_to_extract:
    split_archive_path = os.path.join(MELD_RAW_DIR, f'{split}.tar.gz')
    split_extract_dir  = os.path.join(MELD_RAW_DIR, split) # Extract into MELD.Raw/train/, etc.
    if os.path.exists(split_archive_path):
        if not os.path.exists(split_extract_dir):
            os.makedirs(split_extract_dir)
        print(f"  Extracting {split}.tar.gz to {split_extract_dir}...")
        # Use tar command directly for nested extraction
        !tar -xzf {split_archive_path} -C {split_extract_dir}
    else:
        print(f"  {split_archive_path} not found, skipping extraction for {split} split.")


# ── Inspect structure ─────────────────────────────────────────────────────
# This prints the actual folder tree so we can see what was extracted
print('\nInspecting extracted directory structure:')
# Now inspect MELD_RAW_DIR instead of DATA_DIR to see the actual content
inspect_extracted_structure(MELD_RAW_DIR)

# ── Find MELD root ────────────────────────────────────────────────────────
# Automatically locates the directory containing train_sent_emo.csv
meld_root = find_meld_root(MELD_RAW_DIR) # Pass MELD_RAW_DIR here
print(f'\nMELD root directory: {meld_root}')

# ── Find video directories ────────────────────────────────────────────────
# These should now be MELD_RAW_DIR/train/train_splits/dialogue_videos/ (or similar)
video_dirs = {
    'train': find_video_dir(meld_root, 'train'), # meld_root is now MELD_RAW_DIR
    'dev':   find_video_dir(meld_root, 'dev'),
    'test':  find_video_dir(meld_root, 'test'),
}

print(f'\nVideo directories:')
for split, vdir in video_dirs.items():
    status = '✅' if vdir else '⚠️  not found'
    print(f'  {split}: {vdir or status}')

# Store for later cells
MELD_ROOT  = meld_root
VIDEO_DIRS = video_dirs
print('\n✅ MELD paths resolved.')

---
## Section 07 — Download Official MELD Annotations

**Why:** The official MELD annotation CSV files come from the `declare-lab/MELD` GitHub  
repository. We clone it as a backup, in case the archive's CSVs are outdated.

If the archive already contained valid CSVs (confirmed in Section 06),  
we will use those. The cloned repo is just a reference.

In [ ]:
import os
from training.src.download import clone_annotation_repo

# Clone the official MELD annotation repository
annotation_repo = clone_annotation_repo(force=False)
print(f'Annotation repo: {annotation_repo}')

# List what CSVs are available
for root, dirs, files in os.walk(annotation_repo):
    for f in files:
        if f.endswith('.csv'):
            fpath = os.path.join(root, f)
            size_kb = os.path.getsize(fpath) / 1024
            print(f'  Found: {fpath}  ({size_kb:.0f} KB)')

print('\n✅ Official annotations available.')

---
## Section 08 — Validate Dataset

**Why:** Before training, we verify that:
- All CSV files can be loaded
- Row counts match expected MELD split sizes  
- All emotion/sentiment labels are recognised
- No duplicate IDs exist
- Video files exist and map to CSV rows

A `dataset_validation_report.json` is saved to your Google Drive.

**Expected time:** 2–5 minutes (longer if `check_corrupt_videos=True`)

In [ ]:
from training.src.validate_dataset import validate_dataset
import os # Ensure os is imported

# Redefine meld_root specifically for validation to use the annotations repo
# The annotations repo provides the CSVs directly at a single path: /content/MELD_annotations/data/MELD/
validation_meld_root = os.path.join(annotation_repo, 'data', 'MELD')
print(f'Using validation_meld_root for CSVs: {validation_meld_root}')

# Run full dataset validation
# Set check_corrupt_videos=True to also open a sample of videos with OpenCV
# (adds ~5 minutes but is worth doing at least once)
validation_report = validate_dataset(
    meld_root=validation_meld_root, # Use the annotation repo for CSVs
    video_dirs=VIDEO_DIRS,          # Use the extracted MELD data for videos
    check_corrupt_videos=False,   # Set True for thorough check
    output_dir='/content/drive/MyDrive/AffectraAI/training_outputs',
)

print(f'\nOverall status: {validation_report["overall_status"]}')

---
## Section 09 — Run Smoke Test

**Why:** Before training on all ~10K samples, we run a quick smoke test  
with only 100 train / 30 dev / 30 test samples.

This verifies:
- Dataset loading works
- All three encoders can extract features
- Feature caching works
- Fusion model forward pass succeeds
- One training epoch completes without errors
- Evaluation runs correctly

**The loss and metrics from the smoke test do NOT matter** —  
we only need to confirm there are no crashes.

**Expected time:** 5–10 minutes

⚠️ **IMPORTANT**: After this section, there is a STOP. You must confirm  
that the smoke test passed before proceeding to full training.

In [ ]:
import sys
import torch
import os # Ensure os is imported here as well

# ── Set up ────────────────────────────────────────────────────────────────
from training.src.config import (
    SMOKE_TRAIN_N, SMOKE_DEV_N, SMOKE_TEST_N,
    BATCH_SIZE, RANDOM_SEED,
    MELD_TRAIN_CSV, MELD_DEV_CSV, MELD_TEST_CSV # Import CSV filenames
)
from training.src.utils import set_seed, get_device
from training.src.dataset import load_meld_metadata, MELDCachedDataset, build_dataloader
from training.src.feature_extractors import TextExtractor, AudioExtractor, VideoExtractor, build_video_paths
from training.src.fusion_model import build_model, build_weighted_loss
from training.src.train import train_one_epoch
from training.src.evaluate import evaluate

set_seed(RANDOM_SEED)
device = get_device()
print(f'Device: {device}')
print(f'Smoke test sizes: train={SMOKE_TRAIN_N}, dev={SMOKE_DEV_N}, test={SMOKE_TEST_N}')
print()

# Define the correct base path for MELD annotation CSVs (from the annotation repo)
# This is consistent with how `validation_meld_root` was defined in code-08.
ANNOTATION_CSV_ROOT = os.path.join(annotation_repo, 'data', 'MELD')
print(f'Using annotation CSVs from: {ANNOTATION_CSV_ROOT}')

# ── Load metadata (limited to smoke sizes) ────────────────────────────────
train_df = load_meld_metadata(os.path.join(ANNOTATION_CSV_ROOT, MELD_TRAIN_CSV), max_samples=SMOKE_TRAIN_N)
dev_df   = load_meld_metadata(os.path.join(ANNOTATION_CSV_ROOT, MELD_DEV_CSV),   max_samples=SMOKE_DEV_N)

# ── Extract text features for smoke samples ───────────────────────────────
print('[SMOKE] Extracting text features...')
text_extractor = TextExtractor(device=device, batch_size=32)
train_text_feats = text_extractor.extract_batch(train_df['Utterance'].tolist())
dev_text_feats   = text_extractor.extract_batch(dev_df['Utterance'].tolist())
print(f'  Train text features: {train_text_feats.shape}')
del text_extractor
torch.cuda.empty_cache()

# ── Extract audio features (with zero fallback for missing) ──────────────
print('[SMOKE] Extracting audio features...')
audio_extractor = AudioExtractor(device=device)
train_video_paths = build_video_paths(train_df, VIDEO_DIRS.get('train'))
dev_video_paths   = build_video_paths(dev_df,   VIDEO_DIRS.get('dev'))
train_audio_feats = audio_extractor.extract_batch(train_video_paths, desc='Smoke audio (train)')
dev_audio_feats   = audio_extractor.extract_batch(dev_video_paths,   desc='Smoke audio (dev)')
print(f'  Train audio features: {train_audio_feats.shape}')
del audio_extractor
torch.cuda.empty_cache()

# ── Extract video features ────────────────────────────────────────────────
print('[SMOKE] Extracting video features...')
video_extractor   = VideoExtractor(device=device)
train_video_feats = video_extractor.extract_batch(train_video_paths, desc='Smoke video (train)')
dev_video_feats   = video_extractor.extract_batch(dev_video_paths,   desc='Smoke video (dev)')
print(f'  Train video features: {train_video_feats.shape}')
del video_extractor
torch.cuda.empty_cache()

# ── Build datasets and loaders ────────────────────────────────────────────
train_dataset = MELDCachedDataset(train_df, train_text_feats, train_audio_feats, train_video_feats)
dev_dataset   = MELDCachedDataset(dev_df,   dev_text_feats,   dev_audio_feats,   dev_video_feats)

train_loader = build_dataloader(train_dataset, batch_size=min(16, SMOKE_TRAIN_N), shuffle=True)
dev_loader   = build_dataloader(dev_dataset,   batch_size=min(16, SMOKE_DEV_N),   shuffle=False)

# ── Build model and loss ──────────────────────────────────────────────────
smoke_model = build_model(device)
from collections import Counter
emo_counts  = dict(Counter(train_df['emotion_norm'].tolist()))
sent_counts = dict(Counter(train_df['sentiment_norm'].tolist()))
emo_crit, sent_crit, alpha, beta = build_weighted_loss(emo_counts, sent_counts, device)

# ── One training epoch ────────────────────────────────────────────────────
print('[SMOKE] Running one training epoch...')
import torch.optim as optim
optimizer = optim.AdamW(smoke_model.parameters(), lr=2e-4)
train_metrics = train_one_epoch(
    smoke_model, train_loader, optimizer,
    emo_crit, sent_crit, alpha, beta,
    device=device, scaler=None,
)
print(f'  Train loss: {train_metrics["loss"]:.4f}')

# ── One evaluation run ────────────────────────────────────────────────────
print('[SMOKE] Running evaluation...')
dev_metrics = evaluate(smoke_model, dev_loader, device, split='dev')
print(f'  Dev emotion weighted F1: {dev_metrics["emotion"]["weighted_f1"]:.4f}')

print()
print('=' * 55)
print('✅ SMOKE TEST PASSED — full pipeline works end-to-end!')
print('='* 55)

# Clean up smoke test model
del smoke_model
torch.cuda.empty_cache()

---
## ⛔ STOP — User Confirmation Required

Before proceeding to full training, confirm:

- [ ] The smoke test cell above completed **without errors**
- [ ] You saw `✅ SMOKE TEST PASSED` printed
- [ ] Google Drive is mounted (Section 01 ran successfully)
- [ ] You have at least **3–4 hours** of Colab session time available

**If smoke test failed:**
- Check the error message in the output above
- Common issues: GPU OOM (reduce `BATCH_SIZE` in config.py), missing FFmpeg, import error
- Fix the issue, then re-run from Section 09

**When ready:** Run the cell below to confirm, then continue to Section 10.

In [ ]:
# ⛔ USER CONFIRMATION — Run this cell to explicitly confirm smoke test passed
# Do NOT skip this step.

confirmation = input(
    'Did the smoke test pass? Type YES to continue to full training: '
).strip().upper()

if confirmation == 'YES':
    print('✅ Confirmed. Proceeding to full feature extraction...')
    SMOKE_TEST_PASSED = True
else:
    print('❌ Confirmation not received. Please fix any issues before continuing.')
    SMOKE_TEST_PASSED = False

---
## Section 10 — Extract and Cache Full Features

**Why:** The three pretrained encoders are large (~82M–95M parameters each).  
Running them on all ~10K training samples every epoch would be extremely slow.

Instead, we run each encoder **once** and save the outputs as `.pt` files.  
During training, we only load cached tensors — the encoders are never re-run.

**Feature files saved to Google Drive:**
```
AffectraAI/feature_cache/
  text_train.pt     (9989, 768)  ~29 MB
  text_dev.pt       (1109, 768)  ~3 MB
  text_test.pt      (2610, 768)  ~8 MB
  audio_train.pt    (9989, 768)  ~29 MB
  video_train.pt    (9989, 768)  ~29 MB
  ...etc
```

**If a cache already exists, it will be skipped.**  
Safe to re-run if the session was interrupted.

**Expected times:**
- Text:  15–25 minutes
- Audio: 40–60 minutes  
- Video: 60–90 minutes

In [ ]:
assert SMOKE_TEST_PASSED, 'Smoke test must pass before full feature extraction.'

import os
import torch
import pandas as pd

from training.src.config import (
    MELD_TRAIN_CSV, MELD_DEV_CSV, MELD_TEST_CSV, RANDOM_SEED,
)
from training.src.utils import set_seed, get_device, phase_guard
from training.src.dataset import load_meld_metadata
from training.src.feature_extractors import TextExtractor, AudioExtractor, VideoExtractor, build_video_paths
from training.src.feature_cache import save_features, load_features, cache_status

set_seed(RANDOM_SEED)
device = get_device()
CACHE_DIR = '/content/drive/MyDrive/AffectraAI/feature_cache'

# ── Load full metadata ────────────────────────────────────────────────────
print('Loading full MELD metadata...')
# Use ANNOTATION_CSV_ROOT for loading metadata CSVs, as established in code-08 and code-09
train_df = load_meld_metadata(os.path.join(ANNOTATION_CSV_ROOT, MELD_TRAIN_CSV))
dev_df   = load_meld_metadata(os.path.join(ANNOTATION_CSV_ROOT, MELD_DEV_CSV))
test_df  = load_meld_metadata(os.path.join(ANNOTATION_CSV_ROOT, MELD_TEST_CSV))
print(f'  train: {len(train_df)}, dev: {len(dev_df)}, test: {len(test_df)}')

# ── Phase A: Text features ────────────────────────────────────────────────
if not phase_guard('Text features (train)', os.path.join(CACHE_DIR, 'text_train.pt')):
    print('\n--- Extracting TEXT features ---')
    text_enc = TextExtractor(device=device, batch_size=64)
    for split, df in [('train', train_df), ('dev', dev_df), ('test', test_df)]:
        feats = text_enc.extract_batch(df['Utterance'].tolist(), )
        save_features(feats, 'text', split, cache_dir=CACHE_DIR,
                      sample_ids=df['sample_id'].tolist())
    del text_enc
    torch.cuda.empty_cache()
    print('✅ Text features cached!')

# ── Phase B: Audio features ───────────────────────────────────────────────
if not phase_guard('Audio features (train)', os.path.join(CACHE_DIR, 'audio_train.pt')):
    print('\n--- Extracting AUDIO features ---')
    audio_enc = AudioExtractor(device=device)
    for split, df in [('train', train_df), ('dev', dev_df), ('test', test_df)]:
        vpaths = build_video_paths(df, VIDEO_DIRS.get(split))
        feats  = audio_enc.extract_batch(vpaths, desc=f'Audio ({split})')
        save_features(feats, 'audio', split, cache_dir=CACHE_DIR,
                      sample_ids=df['sample_id'].tolist())
    del audio_enc
    torch.cuda.empty_cache()
    print('✅ Audio features cached!')

# ── Phase C: Video features ───────────────────────────────────────────────
if not phase_guard('Video features (train)', os.path.join(CACHE_DIR, 'video_train.pt')):
    print('\n--- Extracting VIDEO features ---')
    video_enc = VideoExtractor(device=device)
    for split, df in [('train', train_df), ('dev', dev_df), ('test', test_df)]:
        vpaths = build_video_paths(df, VIDEO_DIRS.get(split))
        feats  = video_enc.extract_batch(vpaths, desc=f'Video ({split})')
        save_features(feats, 'video', split, cache_dir=CACHE_DIR,
                      sample_ids=df['sample_id'].tolist())
    del video_enc
    torch.cuda.empty_cache()
    print('✅ Video features cached!')

# ── Summary ───────────────────────────────────────────────────────────────
print('\n=== Cache Status ===')
cache_status(CACHE_DIR)


---
## Section 11 — Train Fusion Model

**Why:** Now we train the lightweight GatedMultimodalFusion model  
on the pre-cached features. Only ~594K parameters are trained.

**Expected time:** 20–45 minutes (up to 30 epochs, with early stopping)

**If OOM error:** Reduce `BATCH_SIZE` in `training/src/config.py` from 64 → 32,  
then re-run from this cell.

In [ ]:
import torch
from collections import Counter

from training.src.config import BATCH_SIZE, RANDOM_SEED
from training.src.utils import set_seed, get_device
from training.src.dataset import MELDCachedDataset, build_dataloader
from training.src.feature_cache import load_all_splits
from training.src.fusion_model import build_model, build_weighted_loss
from training.src.train import train

set_seed(RANDOM_SEED)
device = get_device()
CACHE_DIR = '/content/drive/MyDrive/AffectraAI/feature_cache'
CKPT_DIR  = '/content/drive/MyDrive/AffectraAI/checkpoints'

# ── Load cached features ──────────────────────────────────────────────────
print('Loading cached features...')
train_text,  dev_text,  _         = load_all_splits('text',  CACHE_DIR)
train_audio, dev_audio, _         = load_all_splits('audio', CACHE_DIR)
train_video, dev_video, _         = load_all_splits('video', CACHE_DIR)
print('✅ Features loaded.')

# ── Build datasets ────────────────────────────────────────────────────────
train_dataset = MELDCachedDataset(train_df, train_text, train_audio, train_video)
dev_dataset   = MELDCachedDataset(dev_df,   dev_text,   dev_audio,   dev_video)

train_loader = build_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
dev_loader   = build_dataloader(dev_dataset,   batch_size=BATCH_SIZE, shuffle=False)

# ── Build model and loss ──────────────────────────────────────────────────
model = build_model(device)
emo_counts  = dict(Counter(train_df['emotion_norm'].tolist()))
sent_counts = dict(Counter(train_df['sentiment_norm'].tolist()))
emo_crit, sent_crit, alpha, beta = build_weighted_loss(emo_counts, sent_counts, device)

# ── Train ─────────────────────────────────────────────────────────────────
# To resume: set resume_from to the checkpoint_latest.pt path
history = train(
    model=model,
    train_loader=train_loader,
    dev_loader=dev_loader,
    emotion_criterion=emo_crit,
    sentiment_criterion=sent_crit,
    device=device,
    checkpoint_dir=CKPT_DIR,
    resume_from=None,  # Set to f'{CKPT_DIR}/checkpoint_latest.pt' to resume
)

print(f'\n✅ Training complete!')
print(f'Best epoch: {history["best_epoch"]}')
print(f'Best dev emotion weighted F1: {history["best_dev_emotion_weighted_f1"]:.4f}')

---
## Section 12 — Evaluate Development Split

**Why:** Load the best checkpoint (not latest) and run full evaluation  
on the official dev split to see final dev metrics.

In [ ]:
import os, torch
from training.src.fusion_model import build_model
from training.src.utils import get_device, load_checkpoint
from training.src.dataset import MELDCachedDataset, build_dataloader
from training.src.feature_cache import load_all_splits
from training.src.evaluate import evaluate
from training.src.config import BATCH_SIZE

device = get_device()
CACHE_DIR = '/content/drive/MyDrive/AffectraAI/feature_cache'
CKPT_DIR  = '/content/drive/MyDrive/AffectraAI/checkpoints'
BEST_CKPT = os.path.join(CKPT_DIR, 'checkpoint_best.pt')

# Load best model
eval_model = build_model(device)
load_checkpoint(BEST_CKPT, eval_model, device=device)

# Load dev features
_, dev_text,  _ = load_all_splits('text',  CACHE_DIR)
_, dev_audio, _ = load_all_splits('audio', CACHE_DIR)
_, dev_video, _ = load_all_splits('video', CACHE_DIR)

dev_dataset = MELDCachedDataset(dev_df, dev_text, dev_audio, dev_video)
dev_loader  = build_dataloader(dev_dataset, batch_size=BATCH_SIZE, shuffle=False)

dev_metrics = evaluate(
    eval_model, dev_loader, device, split='dev',
    save_path='/content/drive/MyDrive/AffectraAI/training_outputs/dev_metrics.json'
)

print(f'Dev emotion weighted F1: {dev_metrics["emotion"]["weighted_f1"]:.4f}')

---
## Section 13 — Final Test Evaluation

## ⚠️ RUN THIS CELL EXACTLY ONCE

**Why:** The test split is the held-out evaluation set. It must only be  
evaluated once — at the very end, after all hyperparameter decisions are  
locked in. Re-running test evaluation introduces bias.

**Before running this cell, confirm:**
- You are satisfied with dev split performance
- No further hyperparameter changes are planned
- You understand test metrics will be the final reported numbers

The results are saved to `metrics.json` and will be exported with the model.

In [ ]:
# ⚠️ WARNING: Run this cell ONCE only.
import os, torch
from training.src.fusion_model import build_model
from training.src.utils import get_device, load_checkpoint
from training.src.dataset import MELDCachedDataset, build_dataloader
from training.src.feature_cache import load_all_splits
from training.src.evaluate import evaluate
from training.src.config import BATCH_SIZE

device = get_device()
CACHE_DIR = '/content/drive/MyDrive/AffectraAI/feature_cache'
CKPT_DIR  = '/content/drive/MyDrive/AffectraAI/checkpoints'
BEST_CKPT = os.path.join(CKPT_DIR, 'checkpoint_best.pt')

# Confirm with user before running
confirm = input('Type YES to run final TEST evaluation (this should only run once): ').strip().upper()
assert confirm == 'YES', 'Test evaluation cancelled.'

# Load best model
test_model = build_model(device)
load_checkpoint(BEST_CKPT, test_model, device=device)

# Load test features
_, _, test_text  = load_all_splits('text',  CACHE_DIR)
_, _, test_audio = load_all_splits('audio', CACHE_DIR)
_, _, test_video = load_all_splits('video', CACHE_DIR)

test_dataset = MELDCachedDataset(test_df, test_text, test_audio, test_video)
test_loader  = build_dataloader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

TEST_METRICS = evaluate(
    test_model, test_loader, device, split='test',
    save_path='/content/drive/MyDrive/AffectraAI/training_outputs/test_metrics.json'
)

print(f'\n🏁 FINAL TEST RESULTS:')
print(f'  Emotion weighted F1:   {TEST_METRICS["emotion"]["weighted_f1"]:.4f}')
print(f'  Sentiment weighted F1: {TEST_METRICS["sentiment"]["weighted_f1"]:.4f}')

---
## Section 14 — Export Final Model

**Why:** Export all artifacts needed for the FastAPI backend:
- `model_state.pt` — fusion model weights
- `model_config.json` — architecture config
- `emotion_labels.json`, `sentiment_labels.json` — label mappings
- `metrics.json` — test results
- `text_encoder/` — DistilRoBERTa tokenizer

After exporting, download `affectra_multimodal/` from Colab or Drive  
and place it at `Affectra-AI/models/affectra_multimodal/`.

In [ ]:
import os, torch
from training.src.fusion_model import build_model
from training.src.utils import get_device, load_checkpoint
from training.src.export_model import export_all

device = get_device()
CKPT_DIR  = '/content/drive/MyDrive/AffectraAI/checkpoints'
BEST_CKPT = os.path.join(CKPT_DIR, 'checkpoint_best.pt')

# Local Colab export path (then mirrored to Drive)
LOCAL_EXPORT = '/content/Affectra-AI/models/affectra_multimodal'

# Load best model
export_model = build_model(device)
load_checkpoint(BEST_CKPT, export_model, device=device)

# Export all artifacts
artifact_paths = export_all(
    model=export_model,
    test_metrics=TEST_METRICS,
    output_dir=LOCAL_EXPORT,
    also_save_to_drive=True,  # Mirror to Drive as backup
)

print('\n=== Exported Artifacts ===')
for name, path in artifact_paths.items():
    print(f'  {name}: {path}')

print()
print('=== What to do next ===')
print('1. Download models/affectra_multimodal/ from Colab Files panel')
print('   (or from Drive: AffectraAI/training_outputs/affectra_multimodal/)')
print('2. Place the folder at: Affectra-AI/models/affectra_multimodal/')
print('3. Update MODEL_DIR in .env to: ./models/affectra_multimodal/')
print('4. The FastAPI backend will load model_state.pt from this directory.')